# checking ABY data

Sanitised source snapshot. See the repository README and WORKFLOW.md before running. Outputs and execution metadata are removed. Historical experiments are not an unattended pipeline or reproduced scientific result.


In [ ]:
import os
from pathlib import Path

REPO_ROOT = next(parent for parent in (Path.cwd(), *Path.cwd().parents) if (parent / "prepare_wells.py").is_file())
WORKSPACE_ROOT = Path(os.environ.get("CGWB_WORKSPACE", str(REPO_ROOT / "data" / "cgwb"))).expanduser().resolve()
WORKSPACE_ROOT.mkdir(parents=True, exist_ok=True)
os.chdir(WORKSPACE_ROOT)


In [ ]:
import geopandas as gpd

In [ ]:
pickle_path = f'{WORKSPACE_ROOT}/11. analyzing trends in CGWB and calculated data/input/thresholded_mws_with_CWD_and_WD.pickle'

In [ ]:
import pickle

In [ ]:
# Load the pickle file
with open(pickle_path, 'rb') as file:
    cgwb_data = pickle.load(file)

In [ ]:
cgwb_data

In [ ]:
for column in cgwb_data.columns:
    print(column)

In [ ]:
cgwb_data['geo']

# INDIA STATE BOUNDARY

In [ ]:
import geopandas as gpd

# Path to the GeoJSON file
geojson_path = f'{WORKSPACE_ROOT}/12. Checking WRIS data/input/Indian_states_boundary.geojson'

# Read the GeoJSON file into a GeoDataFrame
india_state_boundary_map = gpd.read_file(geojson_path)

# Display the first few rows of the GeoDataFrame

print(india_state_boundary_map.head())

# Optionally, display the GeoDataFrame's structure
print(india_state_boundary_map.info())


In [ ]:
import matplotlib.pyplot as plt

# Plot the GeoDataFrame
india_state_boundary_map.plot(figsize=(10, 10), edgecolor='k')

# Set the title of the plot
plt.title('Indian States Boundary')

# Show the plot
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import geopandas as gpd
import pandas as pd
import json
from geopy.distance import geodesic

In [ ]:
# Extract and round coordinates from the pickle data
cgwb_data['latitude'] = cgwb_data['geo'].apply(lambda x: round(json.loads(x)['coordinates'][1], 6))
cgwb_data['longitude'] = cgwb_data['geo'].apply(lambda x: round(json.loads(x)['coordinates'][0], 6))

In [ ]:
def find_nearest_point(lat, lon, gdf):
    min_distance = float('inf')
    nearest_point = None
    for idx, row in gdf.iterrows():
        try:
            # Ensure correct order of latitude and longitude
            distance = geodesic((lat, lon), (row['latitude'], row['longitude'])).meters
            if distance < min_distance:
                min_distance = distance
                nearest_point = (row['latitude'], row['longitude'], distance)
        except ValueError as e:
            print(f"Invalid coordinates encountered: {e}")
    # Return the nearest point or default to the original point if none found
    return nearest_point if nearest_point is not None else (lat, lon, 0)

In [ ]:
import pandas as pd

# ABY DATA

In [ ]:
csv_path=f'{WORKSPACE_ROOT}/12. Checking WRIS data/input/Atal_Jal_Disclosed_Ground_Water_Level-2015-2022.csv'

In [ ]:
aby_data=pd.read_csv(csv_path,encoding='latin-1')

In [ ]:
aby_data

In [ ]:
# Accessing both 'Latitude' and 'Longitude' columns
lat_long_data = aby_data[['Latitude', 'Longitude']]

# Display the result
print(lat_long_data)


In [ ]:
# Accessing both 'Latitude' and 'Longitude' columns
lat_long_data = aby_data[['Latitude', 'Longitude']]

# Display the result
print(lat_long_data)

In [ ]:
wd_data = aby_data[['Well_Depth (meters)']]

In [ ]:
wd_data

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Remove NaN values from the 'Well_Depth (meters)' column
wd_data_clean = wd_data.dropna()

# Create a box plot with exchanged axes and inverted well depth values
plt.figure(figsize=(8, 6))
sns.boxplot(data=wd_data_clean, y='Well_Depth (meters)', orient='h')
plt.title('Box Plot of Well Depth (meters)')
plt.xlabel('Well Depth (meters)')
plt.ylabel('')
plt.gca().invert_yaxis()  # Invert the x-axis
plt.show()

# Create a violin plot with exchanged axes and inverted well depth values
plt.figure(figsize=(8, 6))
sns.violinplot(data=wd_data_clean, y='Well_Depth (meters)', orient='h', inner='quartile')
plt.title('Violin Plot of Well Depth (meters)')
plt.xlabel('Well Depth (meters)')
plt.ylabel('')
plt.gca().invert_yaxis()  # Invert the x-axis
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Remove NaN values from the 'Well_Depth (meters)' column
wd_data_clean = wd_data.dropna()

# Define the bin edges with a spacing of 20 meters
bins = np.arange(0, wd_data_clean['Well_Depth (meters)'].max() + 20, 20)

# Bin the well depth data
wd_data_clean['Binned Depth'] = pd.cut(wd_data_clean['Well_Depth (meters)'], bins)

# Create a bar plot
plt.figure(figsize=(10, 6))
sns.countplot(data=wd_data_clean, y='Binned Depth', order=wd_data_clean['Binned Depth'].value_counts().index)
plt.title('Distribution of Well Depths (meters) with 20m Intervals')
plt.xlabel('Count')
plt.ylabel('Well Depth Range (meters)')
plt.gca().invert_yaxis()  # Invert the y-axis to have deeper wells at the bottom
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Remove NaN values from the 'Well_Depth (meters)' column
wd_data_clean = wd_data.dropna()

# Define the bin edges with a spacing of 20 meters
bins = np.arange(0, wd_data_clean['Well_Depth (meters)'].max() + 20, 20)

# Create the histogram
plt.figure(figsize=(12, 8))
counts, bin_edges, _ = plt.hist(wd_data_clean['Well_Depth (meters)'], bins=bins, edgecolor='black', orientation='horizontal', color='skyblue')

# Invert the y-axis to have deeper wells at the bottom
plt.gca().invert_yaxis()

# Set titles and labels
plt.title('Histogram of Well Depths (meters) with 20m Intervals')
plt.ylabel('Well Depth (meters)')
plt.xlabel('Count')

# Annotate each bin with count and percentage
total = sum(counts)
for count, edge in zip(counts, bin_edges[:-1]):
    percentage = (count / total) * 100
    plt.text(count + 1, edge + 10, f'{int(count)} ({percentage:.1f}%)', va='center')

# Set the y-ticks to show only the bin edges
plt.yticks(bin_edges)

# Show the plot
plt.show()

In [ ]:
for column in aby_data.columns:
  print(column)

In [ ]:
# Remove the string '(meters below ground level)' from the column names
aby_data.columns = aby_data.columns.str.replace(' \(meters below ground level\)', '', regex=True)

# Display the updated column names to verify
print(aby_data.columns)

In [ ]:
rows = len(aby_data)
print(rows)

In [ ]:
# Count the number of NaN values in each column
nan_counts = aby_data.isna().sum()

# Display the result
print(nan_counts)

In [ ]:
# Count the number of NaN values in each column
nan_counts = aby_data.isna().sum()

# Calculate the percentage of NaN values in each column
nan_percentage = (nan_counts / len(aby_data)) * 100

# Display the result
print(nan_percentage)

In [ ]:
import matplotlib.pyplot as plt

# Calculate the percentage of NaN values in each column
nan_percentage = (aby_data.isna().sum() / len(aby_data)) * 100

# Plot a bar chart
plt.figure(figsize=(10, 6))
nan_percentage.plot(kind='bar', color='skyblue')

# Add titles and labels
plt.title('Percentage of NaN Values in Each Column', fontsize=10)
plt.xlabel('Columns', fontsize=12)
plt.ylabel('Percentage of NaN Values (%)', fontsize=10)

# Display the plot with corrected xticks
plt.xticks(rotation=90, fontsize=8)  # Removed the extra comma
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Clean up the column names by removing '(meters below ground level)'
aby_data.columns = aby_data.columns.str.replace(' \(meters below ground level\)', '', regex=True)

# List of columns to plot after removing '(meters below ground level)' from column names
columns_to_plot = [
    'Pre-monsoon_2015',
    'Post-monsoon_2015',
    'Pre-monsoon_2016',
    'Post-monsoon_2016',
    'Pre-monsoon_2017',
    'Post-monsoon_2017',
    'Pre-monsoon_2018',
    'Post-monsoon_2018',
    'Pre-monsoon_2019',
    'Post-monsoon_2019',
    'Pre-monsoon_2020',
    'Post-monsoon_2020',
    'Pre-monsoon_2021',
    'Post-monsoon_2021',
    'Pre-monsoon_2022',
    'Post-monsoon_2022'
]

# Drop NaN values for these columns
filtered_data = aby_data[columns_to_plot].dropna()

# Create a box plot
plt.figure(figsize=(12, 8))
sns.boxplot(data=filtered_data)
plt.xticks(rotation=90)  # Rotate x-axis labels for better readability
plt.title('Box Plot of Water Levels Across Years')
plt.ylabel('Meters Below Ground Level')
plt.xlabel('Year and Season')
plt.gca().invert_yaxis()  # Invert the y-axis
plt.show()

# Create a violin plot
plt.figure(figsize=(12, 8))
sns.violinplot(data=filtered_data, inner="quartile")
plt.xticks(rotation=90)  # Rotate x-axis labels for better readability
plt.title('Violin Plot of Water Levels Across Years')
plt.ylabel('Meters Below Ground Level')
plt.xlabel('Year and Season')
plt.gca().invert_yaxis()  # Invert the y-axis
plt.show()

In [ ]:
# List of columns you're interested in
columns_of_interest = [
    'State_Name_With_LGD_Code',
    'District_Name_With_LGD_Code',
    'Block_Name_With_LGD_Code',
    'GP_Name_With_LGD_Code',
    'TYPE',
    'SOURCE',
    'Aquifer'
]

# Loop through each column in the list and print unique values
for column in columns_of_interest:
    print(f"Unique values in column '{column}':")
    print(aby_data[column].unique())
    print("\n" + "-"*50 + "\n")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# Define the colors for each SOURCE type
color_mapping = {'SGWD': 'red', 'CGWB': 'green', 'Atal Jal': 'blue'}


# Plotting the distribution of SOURCE types
fig, ax = plt.subplots(1, 1, figsize=(10, 10))

india_state_boundary_map.plot(ax=ax, edgecolor='black', color='lightgrey')

# Scatter plot for Latitude and Longitude colored by SOURCE
plt.scatter(aby_data['Longitude'], aby_data['Latitude'],
            c=aby_data['SOURCE'].map(color_mapping),
            alpha=0.6, s=10, edgecolor='black', linewidth=0.5)

# Set limits to focus on India
plt.xlim(68, 98)
plt.ylim(20, 38)

# Create custom legend
sgwd_patch = mpatches.Patch(color='red', label='SGWD')
cgwb_patch = mpatches.Patch(color='green', label='CGWB')
atal_jal_patch = mpatches.Patch(color='blue', label='Atal Jal')
plt.legend(handles=[sgwd_patch, cgwb_patch, atal_jal_patch], loc='upper right')

# Add title and labels
plt.title('Distribution of SOURCE Types')
plt.xlabel('Longitude')
plt.ylabel('Latitude')

# Display grid
plt.grid(True)

# Show the plot
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# Define the colors for each SOURCE type
color_mapping = {'SGWD': 'red', 'CGWB': 'green', 'Atal Jal': 'blue'}

# Create a list of source types
source_types = ['SGWD', 'CGWB', 'Atal Jal']

# Plot each source type individually
for source_type in source_types:
    fig, ax = plt.subplots(1, 1, figsize=(10, 10))

    # Plot the state boundaries
    india_state_boundary_map.plot(ax=ax, edgecolor='black', color='lightgrey')

    # Filter the data for the current source type
    filtered_data = aby_data[aby_data['SOURCE'] == source_type]

    # Scatter plot for the current source type
    plt.scatter(filtered_data['Longitude'], filtered_data['Latitude'],
                c=color_mapping[source_type],
                alpha=0.6, s=10, edgecolor='black', linewidth=0.5)

    # Set limits to focus on India
    plt.xlim(68, 98)
    plt.ylim(20, 38)

    # Create a legend for the current source type
    plt.legend(handles=[mpatches.Patch(color=color_mapping[source_type], label=source_type)],
               loc='upper right')

    # Add title and labels
    plt.title(f'Distribution of SOURCE Type: {source_type}')
    plt.xlabel('Longitude')
    plt.ylabel('Latitude')

    # Display grid
    plt.grid(True)

    # Show the plot
    plt.show()

In [ ]:
# Plotting Latitude and Longitude
plt.figure(figsize=(10, 8))

# Scatter plot for the Latitude and Longitude
plt.scatter(aby_data['Longitude'], aby_data['Latitude'], c='blue', alpha=0.6, s=2)

# Label the axes
plt.xlabel('Longitude')
plt.ylabel('Latitude')

# Title of the plot
plt.title('Latitude and Longitude Scatter Plot')

# Display the grid
plt.grid(True, linestyle='--', alpha=0.5)

# Show the plot
plt.show()

In [ ]:
import matplotlib.pyplot as plt

# Plotting the India state boundaries
fig, ax = plt.subplots(figsize=(10, 10))

# Plot the india_state_boundary_map
india_state_boundary_map.plot(ax=ax, edgecolor='black', color='lightgrey')

# Scatter plot for the Latitude and Longitude from abv_data
ax.scatter(aby_data['Longitude'], aby_data['Latitude'], c='blue', alpha=0.6, s=2, label='Data Points')

# Label the axes
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')

# Title of the plot
ax.set_title('Locations mentioned of well data mentioned in ABY with Indian State Boundaries')

# Display the grid
ax.grid(True, linestyle='--', alpha=0.5)

# Add a legend
ax.legend()

# Show the plot
plt.show()

In [ ]:
# Find unique combinations of TYPE and SOURCE
unique_combinations = aby_data[['TYPE', 'SOURCE']].drop_duplicates()

# Display the unique combinations
print("Unique combinations of TYPE and SOURCE:")
print(unique_combinations)

In [ ]:
# Count the occurrences of each unique combination of TYPE and SOURCE
combination_counts = aby_data.groupby(['SOURCE','TYPE']).size().reset_index(name='count')

# Display the unique combinations with their counts
print("Counts of unique combinations of TYPE and SOURCE:")
print(combination_counts)

In [ ]:
# Filter the combination where SOURCE is 'Atal Jal' and TYPE is 'Piezometer'
atal_jal_piezometer = combination_counts[(combination_counts['SOURCE'] == 'Atal Jal') & (combination_counts['TYPE'] == 'Piezometer')]

# Display the result
print("Combination of SOURCE='Atal Jal' and TYPE='Piezometer':")
print(atal_jal_piezometer)

In [ ]:
# Filter the rows where SOURCE is 'Borewell'
borewell_data = aby_data[aby_data['TYPE'] == 'Bore Well']

# Display the filtered data
print("Data for SOURCE='Atal Jal' and TYPE='Piezometer':")
print(borewell_data)

In [ ]:
# Filter the rows where SOURCE is 'Atal Jal' and TYPE is 'Piezometer'
atal_jal_piezometer_data = aby_data[(aby_data['SOURCE'] == 'Atal Jal') & (aby_data['TYPE'] == 'Piezometer')]

# Display the filtered data
print("Data for SOURCE='Atal Jal' and TYPE='Piezometer':")
print(atal_jal_piezometer_data)


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plotting the bar chart
plt.figure(figsize=(10, 6))
sns.barplot(x='count', y='SOURCE', hue='TYPE', data=combination_counts, palette='tab20')

# Add labels and title
plt.xlabel('Count')
plt.ylabel('SOURCE')
plt.title('Counts of Unique SOURCE and TYPE Combinations')

# Show the plot
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd

# Group the data by 'SOURCE' and 'TYPE'
grouped = aby_data.groupby(['SOURCE', 'TYPE'])

# Create an empty list to store the results
missing_data_percentage = []

# Iterate over each group and calculate the percentage of missing values for each column
for (source, typ), group in grouped:
    # Calculate the percentage of missing values for each column in the group
    missing_percentage = (group.isna().sum() / len(group)) * 100

    # Add SOURCE, TYPE, and the missing percentage to the result list
    missing_percentage['SOURCE'] = source
    missing_percentage['TYPE'] = typ
    missing_data_percentage.append(missing_percentage)

# Convert the list to a dataframe
missing_data_percentage_df = pd.DataFrame(missing_data_percentage)

# Reorder the columns to have 'SOURCE' and 'TYPE' first
cols = ['SOURCE', 'TYPE'] + [col for col in missing_data_percentage_df.columns if col not in ['SOURCE', 'TYPE']]
missing_data_percentage_df = missing_data_percentage_df[cols]

# Display the result
print("Percentage of missing data per column for each combination of TYPE and SOURCE:")
print(missing_data_percentage_df)


In [ ]:
import geopandas as gpd
import plotly.express as px

# Assuming aby_data is a GeoDataFrame or has latitude and longitude columns directly
# If aby_data is not a GeoDataFrame with geometry, you first need to create geometries
if not isinstance(aby_data, gpd.GeoDataFrame):
    aby_data = gpd.GeoDataFrame(aby_data, geometry=gpd.points_from_xy(aby_data.Longitude, aby_data.Latitude))

# Filter aby_data for regions above Madhya Pradesh using geographic coordinates
aby_data_above_MP = aby_data[
    (aby_data.geometry.y >= 21) & (aby_data.geometry.y <= 56) &
    (aby_data.geometry.x >= 30) & (aby_data.geometry.x <= 100)
]

# Create a DataFrame with counts of each SOURCE and TYPE combination for the filtered data
combination_counts_above_MP = aby_data_above_MP.groupby(['SOURCE', 'TYPE']).size().reset_index(name='count')


In [ ]:
import plotly.express as px
import pandas as pd

In [ ]:
# Create a grouped bar chart
fig = px.bar(
    combination_counts_above_MP,
    x='SOURCE',
    y='count',
    color='TYPE',
    barmode='group',  # Group bars next to each other
    text='count',  # Display counts on the chart
    title='Grouped Bar Chart of SOURCE and TYPE Combinations Above MP'
)

# Show the plot
fig.show()


In [ ]:
import plotly.express as px



# Create a Sunburst chart with SOURCE as the parent and TYPE as the child
fig = px.sunburst(combination_counts,
                  path=['SOURCE', 'TYPE'],  # Define the hierarchy
                  values='count',           # Define the size of each segment
                  color='SOURCE',           # Color by SOURCE
                  title='Sunburst Chart of SOURCE and TYPE Combinations')

# Show the plot
fig.show()


In [ ]:
import plotly.express as px

# Create a Treemap chart with SOURCE as the parent and TYPE as the child
fig = px.treemap(combination_counts,
                 path=['SOURCE', 'TYPE'],  # Define the hierarchy
                 values='count',           # Define the size of each segment
                 color='SOURCE',           # Color by SOURCE
                 title='Treemap of SOURCE and TYPE Combinations')

# Show the plot
fig.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Ensure that the well depth data is in the DataFrame
# Checking if the necessary columns exist in aby_data_above_MP
print(aby_data_above_MP.columns)

# Create a box plot for each TYPE within each SOURCE
plt.figure(figsize=(12, 8))
ax = sns.boxplot(x='SOURCE', y='Well_Depth (meters)', hue='TYPE', data=aby_data_above_MP)
plt.title('Box Plot of Well Depth by SOURCE and TYPE')
plt.xlabel('SOURCE')
plt.ylabel('Well Depth (meters)')
plt.legend(title='TYPE')
ax.invert_yaxis()  # Invert the y-axis

# Set custom y-axis ticks every 30 meters
y_max = aby_data_above_MP['Well_Depth (meters)'].max()  # Get the max value from your well depth data
y_min = aby_data_above_MP['Well_Depth (meters)'].min()  # Get the min value
ax.set_yticks(np.arange(y_min, y_max, 30))  # Set y-ticks to appear every 30 meters within the range of your data

# Show the plot
plt.show()

In [ ]:
import pandas as pd

# Reading the pickle file
file_path = f'{WORKSPACE_ROOT}/3. Mapping microwatershed with a representative well/input/micro_watershed_gdf.pickle'
micro_watershed_gdf = pd.read_pickle(file_path)

# Display the data
print(micro_watershed_gdf)

In [ ]:
# Convert the DataFrame to a GeoDataFrame
if not isinstance(micro_watershed_gdf, gpd.GeoDataFrame):
    micro_watershed_gdf = gpd.GeoDataFrame(micro_watershed_gdf, geometry='geometry')


In [ ]:
import geopandas as gpd
from shapely.geometry import box

# Define the bounding box coordinates
x_min, y_min, x_max, y_max = 68, 20, 98, 38

# Create a bounding box as a polygon using shapely's box function
bbox = gpd.GeoDataFrame(
    {'geometry': [box(x_min, y_min, x_max, y_max)]},
    crs=micro_watershed_gdf.crs
)

# Perform a spatial join to select only the microwatersheds that lie entirely within the bounding box
selected_microwatersheds = gpd.sjoin(micro_watershed_gdf, bbox, how='inner', predicate='within')

# Display the selected microwatersheds
print(selected_microwatersheds)


In [ ]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point

# Convert the aby_data DataFrame to a GeoDataFrame
aby_data['geometry'] = aby_data.apply(lambda row: Point(row['Longitude'], row['Latitude']), axis=1)
aby_gdf = gpd.GeoDataFrame(aby_data, geometry='geometry')

# Ensure both GeoDataFrames have the same coordinate reference system (CRS)
aby_gdf = aby_gdf.set_crs(epsg=4326)  # Assuming aby_data uses WGS 84 (EPSG:4326)
micro_watershed_gdf = micro_watershed_gdf.to_crs(epsg=4326)

# Perform the spatial join to find where points fall within microwatersheds
merged_data = gpd.sjoin(aby_gdf, micro_watershed_gdf, how='inner', predicate='within')

# Display the merged data
print(merged_data.head())


In [ ]:
merged_data

In [ ]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point

def count_consecutive_non_nan_pairs(series):
    count = 0
    previous_non_nan = pd.notna(series.iloc[0])

    for value in series.iloc[1:]:
        current_non_nan = pd.notna(value)
        if previous_non_nan and current_non_nan:
            count += 1
        previous_non_nan = current_non_nan

    return count

In [ ]:
# List of columns to check for consecutive non-NaN pairs
columns_to_check = [
    'Pre-monsoon_2015', 'Post-monsoon_2015',
    'Pre-monsoon_2016', 'Post-monsoon_2016',
    'Pre-monsoon_2017', 'Post-monsoon_2017',
    'Pre-monsoon_2018', 'Post-monsoon_2018',
    'Pre-monsoon_2019', 'Post-monsoon_2019',
    'Pre-monsoon_2020', 'Post-monsoon_2020',
    'Pre-monsoon_2021', 'Post-monsoon_2021',
    'Pre-monsoon_2022', 'Post-monsoon_2022'
]

# Calculate consecutive non-NaN pairs for each row
merged_data['consecutive_non_nan_pairs'] = merged_data[columns_to_check].apply(count_consecutive_non_nan_pairs, axis=1)

# Sort data by microwatershed_id and consecutive non-NaN pairs
merged_data.sort_values(by=['id', 'consecutive_non_nan_pairs'], ascending=[True, False], inplace=True)

# Drop duplicates, keeping only the first entry for each microwatershed
final_data = merged_data.drop_duplicates(subset=['id'], keep='first')

# Display the final data
print(final_data[['id', 'consecutive_non_nan_pairs']])

In [ ]:
final_data

In [ ]:
import geopandas as gpd

# Assuming 'final_data' already has a 'geometry' column with Point data
if not isinstance(final_data, gpd.GeoDataFrame):
    final_data = gpd.GeoDataFrame(final_data, geometry='geometry')


In [ ]:
import pandas as pd

# Assuming 'final_data' is your DataFrame
final_data = final_data.rename(columns={'id': 'microwatershed_id'})

# Check the new column names to verify the change
print(final_data.columns)

In [ ]:

import matplotlib.pyplot as plt

# Ensure that final_data is a GeoDataFrame
import geopandas as gpd
if not isinstance(final_data, gpd.GeoDataFrame):
    final_data = gpd.GeoDataFrame(final_data, geometry='geometry')

# Plotting
fig, ax = plt.subplots(1, 1, figsize=(10, 10))

# Plot India state boundaries as the background
india_state_boundary_map.plot(ax=ax, edgecolor='black', color='lightgrey')

# Plot points from final_data
final_data.plot(ax=ax, markersize=5, color='blue', marker='o')

# Setting limits might not be necessary depending on your data, but can be adjusted if needed
ax.set_xlim(68, 98)
ax.set_ylim(6, 38)

plt.title('Well Locations Over India Map')
plt.xlabel('Longitude')
plt.ylabel('Latitude')
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import geopandas as gpd

# Ensure that final_data is a GeoDataFrame
if not isinstance(final_data, gpd.GeoDataFrame):
    final_data = gpd.GeoDataFrame(final_data, geometry='geometry')

# Filter the GeoDataFrame to include only wells within the specified coordinates
# Ensure the coordinates align with the latitude and longitude boundaries
wells_above_MP_filtered_gdf = final_data[(final_data.geometry.y >= 21) & (final_data.geometry.y <= 56) &
                                         (final_data.geometry.x >= 30) & (final_data.geometry.x <= 100)]

# Plotting
fig, ax = plt.subplots(1, 1, figsize=(10, 10))

# Plot India state boundaries as the background
india_state_boundary_map.plot(ax=ax, edgecolor='black', color='lightgrey')

# Plot points from wells_above_MP_filtered_gdf
wells_above_MP_filtered_gdf.plot(ax=ax, markersize=2, color='red', marker='o')  # Color changed to red for visibility

# Optionally set plot limits to focus more closely on the region of interest
ax.set_xlim(68, 98)  # Adjust based on the most relevant longitude range
ax.set_ylim(6, 38)   # Adjust based on the most relevant latitude range

plt.title('Well Locations Above MP on India Map')
plt.xlabel('Longitude')
plt.ylabel('Latitude')
plt.show()


In [ ]:
wells_above_MP_filtered_gdf

In [ ]:
# Check if all microwatershed ids are unique in wells_above_MP_filtered_gdf
are_ids_unique = wells_above_MP_filtered_gdf['microwatershed_id'].is_unique

# Display whether all microwatershed ids are unique
are_ids_unique


In [ ]:
# Assuming wells_above_MP_filtered_gdf is already defined and contains the microwatershed_id column
# Display counts of each microwatershed ID to find duplicates
duplicate_ids = wells_above_MP_filtered_gdf['microwatershed_id'].value_counts()

# Filter to show only those IDs that appear more than once
duplicate_ids = duplicate_ids[duplicate_ids > 1]

# Display the duplicate IDs
print(duplicate_ids)


In [ ]:
aquifer_gdf_path=f'{WORKSPACE_ROOT}/12. Checking WRIS data/input/aquifer_data_after_clubbing_categories.pickle'
loaded_aquifer_gdf=pd.read_pickle(aquifer_gdf_path)


In [ ]:
loaded_aquifer_gdf

In [ ]:
for column in loaded_aquifer_gdf.columns:
    print(column)

In [ ]:
loaded_aquifer_gdf

In [ ]:
import geopandas as gpd

In [ ]:
# # Check columns before renaming and dropping to prevent KeyError
# if 'geometry_y' in wells_with_aquifer_info.columns:
#     wells_with_aquifer_info.rename(columns={'category': 'aquifer_category', 'geometry_y': 'aquifer_geometry'}, inplace=True)
#     wells_with_aquifer_info.drop(columns=['index_right', 'geometry_y'], inplace=True)
# else:
#     print("Column names might have been altered or do not exist.")

# if 'geometry_x' in wells_with_aquifer_info.columns:
#     wells_with_aquifer_info.rename(columns={'geometry_x': 'geometry'}, inplace=True)

# # Verify the changes
# print(wells_with_aquifer_info.columns)
# print(wells_with_aquifer_info.head())


In [ ]:
wells_with_aquifer_info = gpd.sjoin(
    wells_above_MP_filtered_gdf,
    loaded_aquifer_gdf,
    how='left',
    predicate='within',
    lsuffix='wells',
    rsuffix='aquifer'
).copy()


In [ ]:
wells_with_aquifer_info

In [ ]:
for column in wells_with_aquifer_info.columns:
    print(column)

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

# Define specific colors for each aquifer category
category_colors = {
    'Alluvial': '#FFDBAC',         # light sandlewood
    'Himalayan Systems': 'peru',       # mild yellow (skin color)
    'Volcanic Systems': 'mediumseagreen',  # medium green (like grass green)
    'Sedimentary Hard Rock': 'dodgerblue',
    'Sedimentary Soft Rock': 'skyblue',
    'Crystalline Systems': 'lightpink',
    'Unclassified': 'red'  # Adding a color for 'Other' category if it exists
}

# Ensure GeoDataFrames are in the same CRS for accurate overlay
if wells_with_aquifer_info.crs != loaded_aquifer_gdf.crs:
    wells_with_aquifer_info = wells_with_aquifer_info.to_crs(loaded_aquifer_gdf.crs)

# Plotting setup
fig, ax = plt.subplots(figsize=(12, 10))

# Plot the aquifers with the colors defined in the dictionary
for category, color in category_colors.items():
    subset = loaded_aquifer_gdf[loaded_aquifer_gdf['aquifer_category'] == category]
    subset.plot(ax=ax, color=color, edgecolor='black', linewidth=0.1, label=category)

# Overlay the well points using a unique color
wells_with_aquifer_info.plot(ax=ax, marker='o', color='purple', markersize=2, label='Wells')

# Create a custom legend for the aquifer categories and wells
legend_handles = [Patch(facecolor=color, edgecolor='black', label=category) for category, color in category_colors.items()]
legend_handles.append(Patch(facecolor='purple', edgecolor='black', label='Wells'))  # Wells are now purple
plt.legend(handles=legend_handles, title='Legend', loc='upper right', fontsize='small')

# Add titles and labels
plt.title('Aquifer Map with Overlaid Well Locations')
plt.xlabel('Longitude')
plt.ylabel('Latitude')

# Show the plot
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Count the number of wells in each aquifer category
category_counts = wells_with_aquifer_info['aquifer_category'].value_counts().reset_index()
category_counts.columns = ['aquifer_category', 'count']

# Calculate the total number of wells
total_wells = category_counts['count'].sum()

# Calculate the percentage of total for each category
category_counts['percentage'] = (category_counts['count'] / total_wells) * 100

# Display the data to see if it's correctly prepared
print(category_counts)

In [ ]:
# Plotting the grouped bar graph
fig, ax = plt.subplots(figsize=(12, 8))
bars = ax.bar(category_counts['aquifer_category'], category_counts['count'], color=[category_colors.get(x, '#333333') for x in category_counts['aquifer_category']])

# Adding labels and titles
ax.set_xlabel('Aquifer Category')
ax.set_ylabel('Count of Wells')
ax.set_title('Number of Wells per Aquifer Category')
ax.set_xticks(range(len(category_counts['aquifer_category'])))
ax.set_xticklabels(category_counts['aquifer_category'], rotation=45)

# Adding count and percentage on top of each bar
for bar, count, percentage in zip(bars, category_counts['count'], category_counts['percentage']):
    yval = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, yval, f'{int(count)}\n({percentage:.1f}%)', ha='center', va='bottom', fontsize=10)

# Show the plot
plt.show()

# taking only borewell, piezometer and tube well


In [ ]:
filtered_wells = wells_with_aquifer_info[
    (wells_with_aquifer_info['TYPE'].isin(['Piezometer', 'Bore Well', 'Tube Well'])) &
    (wells_with_aquifer_info['aquifer_category'] != 'Unclassified')
]

In [ ]:
filtered_wells

In [ ]:
for column in filtered_wells.columns:
    print(column)

In [ ]:
import os

In [ ]:
# Path where you want to save the file
save_path = f'{WORKSPACE_ROOT}/13. Checking ABY data/output/'

In [ ]:
# Full path for the file
file_path = os.path.join(save_path, 'wells_before_checking_NANs.pkl')

# Save the filtered_wells object
with open(file_path, 'wb') as file:
    pickle.dump(filtered_wells, file)

print(f'Successfully saved filtered_wells at: {file_path}')



1.   Reading the pickle



In [ ]:
import pickle

# Path to the saved file
file_path = f'{WORKSPACE_ROOT}/13. Checking ABY data/output/wells_before_checking_NANs.pkl'

# Load the DataFrame (or object) back
with open(file_path, 'rb') as file:
    opening_filtered_wells = pickle.load(file)

In [ ]:
opening_filtered_wells

In [ ]:
for column in opening_filtered_wells.columns:
    print(column)

In [ ]:
unique_values = opening_filtered_wells['consecutive_non_nan_pairs'].unique()
print(unique_values)

In [ ]:
for aquifer, group in opening_filtered_wells.groupby('aquifer_category'):
    unique_values = group['consecutive_non_nan_pairs'].unique()
    print(f"Aquifer Category: {aquifer}")
    print(f"Unique consecutive_non_nan_pairs: {unique_values}")
    print("-" * 50)

In [ ]:
percentiles = opening_filtered_wells.groupby('aquifer_category')['consecutive_non_nan_pairs'].describe(percentiles=[0.2, 0.4, 0.5, 0.75, 1.0])
print(percentiles)

In [ ]:
import pandas as pd

In [ ]:
# Total number of wells before filtering (overall)
total_wells_before = len(opening_filtered_wells)
print(f"Total wells before thresholding (across all aquifer categories): {total_wells_before}")

# Step 1: Compute 75% of max consecutive_non_nan_pairs for each aquifer_category
thresholds = (
    opening_filtered_wells
    .groupby('aquifer_category')['consecutive_non_nan_pairs']
    .max()  # Get max per aquifer_category
    .mul(0.75)  # Compute 75% of max (use 0.75 for 75%)
)

# Step 2: Filter the DataFrame using these thresholds
def meets_threshold(row):
    aquifer = row['aquifer_category']
    threshold = thresholds[aquifer]  # Get threshold for this aquifer
    return row['consecutive_non_nan_pairs'] >= threshold

# Apply filter
filtered_wells_based_on_nans = opening_filtered_wells[opening_filtered_wells.apply(meets_threshold, axis=1)]

# Total number of wells after filtering (overall)
total_wells_after = len(filtered_wells_based_on_nans)

print(f"Total wells after thresholding (across all aquifer categories): {total_wells_after}")
print(f"Overall percentage retained: {100 * total_wells_after/ total_wells_before:.2f}%\n")

# Step 3: Per aquifer_category stats
print("Breakdown per aquifer_category:")

aquifer_stats = []

for aquifer, group in opening_filtered_wells.groupby('aquifer_category'):
    before_count = len(group)
    after_count = len(filtered_wells_based_on_nans[filtered_wells_based_on_nans['aquifer_category'] == aquifer])
    percentage_retained = 100 * after_count / before_count

    aquifer_stats.append({
        'aquifer_category': aquifer,
        'wells_before': before_count,
        'wells_after': after_count,
        'percentage_retained': percentage_retained
    })

# Convert to DataFrame to display neatly
aquifer_stats_df = pd.DataFrame(aquifer_stats)

# Show the result table
print(aquifer_stats_df.to_string(index=False))

# Optional: If you want to save this breakdown as CSV
# aquifer_stats_df.to_csv('[private path removed]

In [ ]:
# Save the filtered wells DataFrame to CSV
filtered_wells_based_on_nans.to_csv(f'{WORKSPACE_ROOT}/13. Checking ABY data/output/filtered_wells_based_on_nans.csv', index=False)

In [ ]:
aquifer_stats_df

In [ ]:
is_unique = filtered_wells_based_on_nans['microwatershed_id'].is_unique

if is_unique:
    print("✅ All microwatershed_id values are unique.")
else:
    print("❌ Some microwatershed_id values are repeated.")

In [ ]:
filtered_wells_based_on_nans['microwatershed_id']

In [ ]:
# Reading the pickle file
file_path = f'{WORKSPACE_ROOT}/3. Mapping microwatershed with a representative well/input/micro_watershed_gdf.pickle'
micro_watershed_gdf = pd.read_pickle(file_path)

In [ ]:
for column in micro_watershed_gdf:
    print(column)

In [ ]:
# Extract unique microwatershed_id from filtered_wells_based_on_nans
selected_ids = filtered_wells_based_on_nans['microwatershed_id'].unique()

# Filter micro_watershed_gdf to match these ids
matched_micro_watershed_gdf = micro_watershed_gdf[micro_watershed_gdf['id'].isin(selected_ids)]

# Keep only microwatershed_id and geometry columns
result_df = matched_micro_watershed_gdf[['id', 'geometry']]

# Result
print(f"Number of matched microwatershed_id: {len(result_df)}")
print(result_df.head())

In [ ]:
# Rename 'id' to 'microwatershed_id'
result_df = result_df.rename(columns={'id': 'microwatershed_id'})

In [ ]:
result_df

In [ ]:
import os

# Define the path
output_dir = f'{WORKSPACE_ROOT}/13. Checking ABY data/output/'

# Define file name
output_path = os.path.join(output_dir, 'deep_aquifer_microwatersheds_with_geometry.csv')

# Save as CSV
result_df.to_csv(output_path, index=False)

print(f'Successfully saved to: {output_path}')


#WRIS data

In [ ]:
import pandas as pd

# Path to the pickle file
pickle_path = f'{WORKSPACE_ROOT}/12. Checking WRIS data/input/mws_without_calculated_cwd.pickle'

# Load the pickle file
mws_data = pd.read_pickle(pickle_path)

# Display the first few rows of the DataFrame to verify it's loaded correctly
print(mws_data.head())


In [ ]:
for column in mws_data.columns:
  print(column)

In [ ]:
aby_data_above_MP

In [ ]:
for column in aby_data_above_MP.columns:
    print(column)

In [ ]:
wells_above_MP_filtered_gdf

In [ ]:
# Ensure the column names are correct and adjust these lines if necessary
mws_ids = set(mws_data['id_micro_watershed'])
aby_ids = set(wells_above_MP_filtered_gdf['microwatershed_id'])

# Find the intersection of both sets to get common IDs
common_ids = mws_ids.intersection(aby_ids)

# Output the common microwatershed IDs
print("Common microwatershed IDs:", common_ids)
print("Number of common IDs:", len(common_ids))


In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt

# Step 1: Filter the data based on common microwatershed IDs
mws_filtered = mws_data[mws_data['id_micro_watershed'].isin(common_ids)]
aby_filtered = wells_above_MP_filtered_gdf[wells_above_MP_filtered_gdf['microwatershed_id'].isin(common_ids)]

# Ensure the CRS is consistent between both datasets
if aby_filtered.crs != india_state_boundary_map.crs:
    aby_filtered = aby_filtered.to_crs(india_state_boundary_map.crs)

# Convert coordinates in mws_filtered to a GeoSeries (assuming 'coordinates' contains (lon, lat) tuples)
mws_filtered['geometry'] = gpd.GeoSeries.from_xy(mws_filtered['coordinates'].apply(lambda x: x[0]),
                                                 mws_filtered['coordinates'].apply(lambda x: x[1]),
                                                 crs='EPSG:4326')

# Convert to GeoDataFrame if not already
mws_filtered_gdf = gpd.GeoDataFrame(mws_filtered, geometry='geometry')

# Step 2: Plotting the filtered data
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 10))

# Plot India state boundaries as the background on both subplots
india_state_boundary_map.plot(ax=ax1, color='lightgrey', edgecolor='black')
india_state_boundary_map.plot(ax=ax2, color='lightgrey', edgecolor='black')

# Plot wells from mws_filtered (CGWB data) on the first subplot
mws_filtered_gdf.plot(ax=ax1, marker='o', color='blue', markersize=20, label='CGWB Wells')

# Plot wells from aby_filtered (ABY data) on the second subplot
aby_filtered.plot(ax=ax2, marker='o', color='green', markersize=20, label='ABY Wells')

# Set titles and legends
ax1.set_title('CGWB Wells (Common Microwatersheds)')
ax1.set_xlabel('Longitude')
ax1.set_ylabel('Latitude')
ax1.legend()

ax2.set_title('ABY Wells (Common Microwatersheds)')
ax2.set_xlabel('Longitude')
ax2.set_ylabel('Latitude')
ax2.legend()

# Adjust layout for better spacing
plt.tight_layout()

# Show the plot
plt.show()


In [ ]:
# Ensure the column names are correct and adjust these lines if necessary
mws_ids = set(mws_data['id_micro_watershed'])
aby_ids = set(wells_above_MP_filtered_gdf['microwatershed_id'])

# Find the intersection of both sets to get common IDs
common_ids = mws_ids.intersection(aby_ids)

# Filter the wells_above_MP_filtered_gdf to only include wells in the common microwatersheds
aby_filtered_common_mws = wells_above_MP_filtered_gdf[wells_above_MP_filtered_gdf['microwatershed_id'].isin(common_ids)]


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Group by SOURCE and TYPE and count the number of occurrences
source_type_counts = aby_filtered_common_mws.groupby(['SOURCE', 'TYPE']).size().reset_index(name='count')

# Create the grouped bar chart
plt.figure(figsize=(12, 8))
ax = sns.barplot(x='SOURCE', y='count', hue='TYPE', data=source_type_counts)

# Adding labels and title
plt.xlabel('Source')
plt.ylabel('Count of Wells')
plt.title('Distribution of SOURCE and TYPE in Common Microwatersheds')
plt.legend(title='Type')

# Annotate bars with the count
for p in ax.patches:
    height = p.get_height()
    ax.annotate(f'{int(height)}',
                (p.get_x() + p.get_width() / 2., height),
                ha = 'center', va = 'bottom',
                fontsize=10)

# Show the plot
plt.show()


In [ ]:
import geopandas as gpd

# Convert mws_data to GeoDataFrame if it's not already one
if 'geometry' not in mws_data.columns:
    mws_data['geometry'] = gpd.GeoSeries.from_xy(mws_data['coordinates'].apply(lambda x: x[0]),
                                                 mws_data['coordinates'].apply(lambda x: x[1]),
                                                 crs='EPSG:4326')  # Assuming WGS 84 (EPSG:4326)

# Extract coordinates for comparison
mws_coords = mws_data['geometry'].apply(lambda geom: (geom.x, geom.y))
aby_coords = wells_above_MP_filtered_gdf['geometry'].apply(lambda geom: (geom.x, geom.y))

# Find common points
common_points = mws_coords.isin(aby_coords)

# Number of common points
num_common_points = common_points.sum()

# Display the result
print(f"Number of common points: {num_common_points}")
